#### **Project:** Queens Supermarket Sales ETL Pipeline  
**Client Context:** Midroc Commerce (Queens Supermarket)  

**Objective:**  
Extract the raw synthetic sales data and ingest it into the Bronze layer following Medallion Architecture best practices. This simulates pulling data from ERP/POS systems.

**Key Steps:**
- Load raw CSV data
- Basic validation
- Save raw data into Bronze layer (as Parquet for better performance)
- Record ingestion metadata

#### Import Libraries

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
import json

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

Libraries imported successfully!


#### Define Configuration & Paths

In [2]:
# Define paths
RAW_DATA_PATH = '../data/raw/queens_supermarket_sales_raw.csv'
BRONZE_PATH = '../data/bronze/'

# Create directories if they don't exist
os.makedirs(BRONZE_PATH, exist_ok=True)

# Ingestion metadata
ingestion_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Configuration and paths set successfully!")
print(f"Raw data path    : {RAW_DATA_PATH}")
print(f"Bronze layer path: {BRONZE_PATH}")

Configuration and paths set successfully!
Raw data path    : ../data/raw/queens_supermarket_sales_raw.csv
Bronze layer path: ../data/bronze/


#### Load Raw Data

In [4]:
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Shape of raw dataset: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")

Shape of raw dataset: (50000, 19)
Columns: ['invoice_id', 'branch_id', 'branch_name', 'city', 'customer_type', 'gender', 'product_line', 'unit_price', 'quantity', 'tax_15', 'total', 'date', 'time', 'payment', 'cogs', 'gross_income', 'gross_margin_percentage', 'rating', 'month']


#### Initial Data Validation

##### Check for missing values

In [5]:
missing_values = df_raw.isnull().sum()
print("Missing Values:")
print(missing_values[missing_values > 0])

Missing Values:
Series([], dtype: int64)


##### Check data types

In [6]:
print("\nData Types:")
print(df_raw.dtypes)


Data Types:
invoice_id                  object
branch_id                   object
branch_name                 object
city                        object
customer_type               object
gender                      object
product_line                object
unit_price                 float64
quantity                     int64
tax_15                     float64
total                      float64
date                        object
time                        object
payment                     object
cogs                       float64
gross_income               float64
gross_margin_percentage    float64
rating                     float64
month                       object
dtype: object


##### Basic statistics

In [7]:
print("\nBasic Statistics:")
print(df_raw.describe().round(2))


Basic Statistics:
       unit_price  quantity   tax_15    total     cogs  gross_income  \
count    50000.00  50000.00 50000.00 50000.00 50000.00      50000.00   
mean       390.19      6.51   380.81  2919.53  1776.14       1143.39   
std        315.96      3.44   401.99  3081.92  1882.01       1219.47   
min         25.04      1.00     3.77    28.89    15.70         10.47   
25%        134.93      4.00    96.64   740.87   447.94        288.00   
50%        292.36      6.00   226.50  1736.52  1053.99        676.29   
75%        592.52      9.00   526.98  4040.22  2457.08       1578.32   
max       1199.99     12.00  2159.69 16557.65 10973.66       7523.78   

       gross_margin_percentage   rating  
count                 50000.00 50000.00  
mean                     39.15     6.75  
std                       4.02     1.59  
min                      32.17     4.00  
25%                      35.67     5.40  
50%                      39.15     6.80  
75%                      42.65     8.1

##### Check for duplicate invoices

In [8]:
duplicates = df_raw.duplicated(subset=['invoice_id']).sum()
print(f"\nDuplicate Invoice IDs: {duplicates}")


Duplicate Invoice IDs: 0


##### Validate date range

In [9]:
df_raw['date'] = pd.to_datetime(df_raw['date'])
print(f"\nDate Range: {df_raw['date'].min()} to {df_raw['date'].max()}")


Date Range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


#### Data Type Optimization & Cleaning

In [10]:
# Convert date column to datetime
df_raw['date'] = pd.to_datetime(df_raw['date'])

# Optimize data types for better performance
df_raw['quantity'] = df_raw['quantity'].astype('int16')
df_raw['branch_id'] = df_raw['branch_id'].astype('category')
df_raw['customer_type'] = df_raw['customer_type'].astype('category')
df_raw['gender'] = df_raw['gender'].astype('category')
df_raw['product_line'] = df_raw['product_line'].astype('category')
df_raw['payment'] = df_raw['payment'].astype('category')

print("Data types optimized and basic cleaning completed.")

Data types optimized and basic cleaning completed.


#### Save to Bronze Layer

In [11]:
bronze_file_path = os.path.join(BRONZE_PATH, 'queens_supermarket_sales_bronze.parquet')

df_raw.to_parquet(bronze_file_path, index=False, compression='snappy')

print(f"Data successfully ingested into Bronze layer!")
print(f"File saved at: {bronze_file_path}")
print(f"File size: {os.path.getsize(bronze_file_path) / (1024*1024):.2f} MB")

Data successfully ingested into Bronze layer!
File saved at: ../data/bronze/queens_supermarket_sales_bronze.parquet
File size: 2.19 MB


#### Save Ingestion Metadata

In [12]:
# Create and save ingestion metadata
metadata = {
    "ingestion_date": ingestion_timestamp,
    "source_file": "queens_supermarket_sales_raw.csv",
    "target_layer": "bronze",
    "records_ingested": len(df_raw),
    "columns": list(df_raw.columns),
    "file_format": "parquet",
    "compression": "snappy"
}

metadata_path = os.path.join(BRONZE_PATH, 'ingestion_metadata.json')

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Ingestion metadata saved at: {metadata_path}")

Ingestion metadata saved at: ../data/bronze/ingestion_metadata.json


#### Final Verification

In [14]:
# Verify the saved Parquet file
df_bronze = pd.read_parquet(bronze_file_path)

print("Verification - Bronze Layer Sample:")
display(df_bronze.head())

print(f"\nBronze layer shape: {df_bronze.shape}")
print("Ingestion Phase completed successfully!")

Verification - Bronze Layer Sample:


,invoice_id,branch_id,branch_name,city,customer_type,gender,product_line,unit_price,quantity,tax_15,total,date,time,payment,cogs,gross_income,gross_margin_percentage,rating,month
0,INV100000,Q001,Queens - Bole,Addis Ababa,Member,Female,Food and Beverages,265.40,5,199.05,1526.05,2025-09-18,14:05,CBE Birr,887.41,638.64,41.85,7.40,2025-09
1,INV100001,Q001,Queens - Bole,Addis Ababa,Member,Female,Health and Beauty,221.93,10,332.90,2552.20,2025-04-21,09:58,Card,1482.98,1069.22,41.89,5.80,2025-04
2,INV100002,Q002,Queens - CMC,Addis Ababa,Member,Male,Food and Beverages,383.42,2,115.03,881.87,2025-12-14,16:49,Cash,545.90,335.97,38.10,7.60,2025-12
3,INV100003,Q002,Queens - CMC,Addis Ababa,Normal,Female,Electronic Accessories,1170.10,2,351.03,2691.23,2025-07-06,08:01,Cash,1778.77,912.46,33.90,4.60,2025-07
4,INV100004,Q001,Queens - Bole,Addis Ababa,Member,Male,Snacks and Confectionery,63.50,6,57.15,438.15,2025-05-31,19:10,Cash,279.91,158.24,36.12,4.40,2025-05



Bronze layer shape: (50000, 19)
Ingestion Phase completed successfully!
